In [ ]:
# -*- coding: utf-8 -*-
"""
SAP T-Code ZSDR0026 계약별 장비/재료 잔액조회 자동화
- 사업장 163개 순회
- 계약진행상태: 전체 선택
- ALV → 클립보드 내보내기 → MySQL DB 저장
"""

import win32com.client
import pyperclip
import pymysql
import time
import sys
from datetime import datetime, date

# ─────────────────────────────────────────────
# 설정값
# ─────────────────────────────────────────────
SAP_TCODE  = "********"

# ▼ 기준일자 설정 (빈 문자열이면 오늘 날짜 자동 사용)
# 형식: "YYYYMMDD"  예) BASE_DATE = "20260315"
BASE_DATE  = "20260315"

# BASE_DATE 가 비어있으면 오늘 날짜로 자동 설정
if not BASE_DATE:
    BASE_DATE = date.today().strftime("%Y%m%d")

# SAP 화면 입력용: YYYY.MM.DD 형식  예) "2026.03.15"
SAP_DATE = datetime.strptime(BASE_DATE, "%Y%m%d").strftime("%Y.%m.%d")

# 사업장 코드 163개 (ZTRR0003과 동일)
PLANT_CODES = [
    '1010','1020','1030','1040','1050','1060','1070','1080','1090',
    '1110','1120','1130','1140','1150','1160','1170','1180','1190',
    '1210','1220','1230','1240','1250','1260','1270','1280','1290',
    '1310','1320','1330','1340','1350','1360','1370','1380','1390',
    '1410','1420','1430','1440','1450','1460','1470','1480','1490',
    '1510','1520','1530','1540','1550','1560','1570','1580','1590',
    '1610','1620','1630','1640','1650','1660','1670','1680','1690',
    '1710','1720','1730','1740','1750','1760','1770','1780','1790',
    '1810','1820','1830','1840','1850','1860','1870','1880','1890',
    '1910','1920','1930','1940','1950','1960','1970','1980','1990',
    '2010','2020','2030','2040','2050','2060','2070','2080','2090',
    '2110','2120','2130','2140','2150','2160','2170','2180','2190',
    '2210','2220','2230','2240','2250','2260','2270','2280','2290',
    '2310','2320','2330','2340','2350','2360','2370','2380','2390',
    '2410','2420','2430','2440','2450','2460','2470','2480','2490',
    '2510','2520','2530','2540','2550','2560','2570','2580','2590',
    '2610','2620','2630','2640','2650','2660','2670','2680','2690',
    '2710','2720','2730','2740','2750','2760','2770','2780','2790',
    '9010','9020',
]

DB_HOST  = "********"
DB_PORT  = 5010
DB_USER  = "********"
DB_PASS  = "********"           # ← 실제 비밀번호 입력
DB_NAME  = "ERP_Translate"
DB_TABLE = "ZSDR0026_잔액조회"

# 데이터가 없는 사업장 스킵 여부
SKIP_EMPTY = True

# DB 배치 사이즈
BATCH_SIZE = 500

# ALV 클립보드 내보내기 후 대기시간(초)
CLIPBOARD_WAIT = 3.0

NUMERIC_COLS = {
    '총계약금액', '총이용금액', '총계약잔액',
    '제품계약금액', '제품이용금액', '제품잔액',
    '장비계약금액', '장비이용금액', '남은장비잔액',
    '재료계약금액', '재료이용금액', '남은재료잔액',
}

# ─────────────────────────────────────────────
# SAP 연결
# ─────────────────────────────────────────────
def connect_sap():
    """SAP GUI COM 연결 반환"""
    sap_gui = win32com.client.GetObject("SAPGUI")
    app = sap_gui.GetScriptingEngine
    conn = app.Children(0)
    session = conn.Children(0)
    print(f"[SAP] 연결 성공: {session.Info.SystemName}")
    return session


# ─────────────────────────────────────────────
# T-Code 이동
# ─────────────────────────────────────────────
def navigate_to_tcode(session, tcode):
    session.findById("wnd[0]/tbar[0]/okcd").text = f"/n{tcode}"
    session.findById("wnd[0]").sendVKey(0)
    time.sleep(1.5)


# ─────────────────────────────────────────────
# 선택화면 입력
# ─────────────────────────────────────────────
def input_conditions(session, plant_code):
    """
    사업장 입력 + 계약진행상태 '전체' 선택
    실제 필드 ID는 field_dump_zsdr0026.py 로 확인 후 수정
    """
    # 기준일자 입력 (field_dump 확인: ctxtP_LFDAT, SAP 형식 YYYY.MM.DD)
    session.findById("wnd[0]/usr/ctxtP_LFDAT").text = SAP_DATE

    # 사업장 필드 ID (field_dump 확인 완료)
    session.findById("wnd[0]/usr/ctxtS_VKBUR-LOW").text = plant_code

    # 계약진행상태 → 전체 선택 (field_dump 확인: radR1=진행, radR2=전체)
    session.findById("wnd[0]/usr/radR2").select()

    time.sleep(0.3)


# ─────────────────────────────────────────────
# F8 실행 & 결과 대기
# ─────────────────────────────────────────────
def execute_report(session):
    session.findById("wnd[0]").sendVKey(8)  # F8
    time.sleep(2.5)


# ─────────────────────────────────────────────
# ALV Grid 탐색
# ─────────────────────────────────────────────
def find_alv_grid(session):
    """
    ZSDR0026 ALV Grid 경로 탐색
    ZTRR0003과 동일하게 DockingContainer 구조일 가능성 높음
    """
    # ALV Grid 경로 (find_alv 확인 완료: DockingContainer 구조)
    try:
        obj = session.findById("wnd[0]/shellcont/shell")
        return obj
    except Exception as e:
        raise RuntimeError(f"ALV Grid를 찾지 못했습니다: {e}")


# ─────────────────────────────────────────────
# 클립보드 내보내기
# ─────────────────────────────────────────────
def extract_alv_to_clipboard(session):
    """ALV → 클립보드 내보내기 → 텍스트 반환"""
    shell = find_alv_grid(session)

    row_count = 0
    try:
        row_count = shell.RowCount
    except Exception:
        pass

    if row_count == 0:
        print("    ↳ 데이터 없음 (RowCount=0)")
        return None

    print(f"    ↳ ALV RowCount={row_count}")

    # 전체 선택: Ctrl+End → Ctrl+Home → Ctrl+Shift+End
    try:
        shell.setCurrentCell(0, shell.ColumnOrder[0])
        shell.sendVKey(11)   # Ctrl+End (마지막으로)
        shell.sendVKey(11)
        shell.sendVKey(999)  # Ctrl+Home
        shell.selectedRows = f"0-{row_count - 1}"
    except Exception:
        pass

    # 클립보드 이전 값 초기화
    try:
        pyperclip.copy("")
    except Exception:
        pass
    time.sleep(0.2)

    # 내보내기 → 클립보드(스프레드시트)
    try:
        shell.pressToolbarContextButton("&MB_EXPORT")
        time.sleep(0.8)
        shell.selectContextMenuItem("&PC")
        time.sleep(0.8)
    except Exception as e:
        raise RuntimeError(f"내보내기 버튼 실행 실패: {e}")

    # 형식 선택 창: 클립보드 = [4,0] (스크립트 레코더 확인 완료)
    try:
        radio = session.findById(
            "wnd[1]/usr/subSUBSCREEN_STEPLOOP:SAPLSPO5:0150"
            "/sub:SAPLSPO5:0150/radSPOPLI-SELFLAG[4,0]"
        )
        radio.select()
        radio.setFocus()
        session.findById("wnd[1]/tbar[0]/btn[0]").press()
        time.sleep(CLIPBOARD_WAIT)
    except Exception as e:
        raise RuntimeError(f"형식 선택 창 처리 실패: {e}")

    # 클립보드 읽기
    text = pyperclip.paste()
    if not text or len(text) < 10:
        raise RuntimeError("클립보드가 비어있습니다.")
    return text


# ─────────────────────────────────────────────
# 클립보드 텍스트 파싱
# ─────────────────────────────────────────────
def _clean_number(s):
    """금액 문자열 → float (음수 처리 포함)"""
    s = s.strip().replace(',', '')
    if not s or s == '0':
        return 0.0
    # SAP 음수 표현: '280,000-' → -280000
    negative = s.endswith('-')
    s = s.rstrip('-').strip()
    try:
        val = float(s)
        return -val if negative else val
    except ValueError:
        return None


def parse_clipboard_text(text, plant_code, run_dt):
    """
    파이프(|) split 방식 파싱 (ZTRR0003과 동일)
    - split('|') → 앞뒤 빈 요소 제거 → strip
    - 빈 셀도 파이프 사이에 공백으로 존재하므로 컬럼 밀림 없음
    """
    lines = text.splitlines()

    # 헤더 줄 찾기
    header_line = None
    header_idx  = None
    for i, line in enumerate(lines):
        if '|' in line and '계약번호' in line:
            header_line = line
            header_idx  = i
            break

    if header_line is None:
        print("    ↳ 헤더를 찾지 못했습니다.")
        return []

    # 헤더 파싱: split('|') → 앞뒤 빈 요소 제거 → strip
    headers = [h.strip() for h in header_line.split('|')]
    headers = [h for h in headers if h]  # 맨 앞/뒤 빈 문자열 제거
    print(f"    ↳ 헤더 컬럼 수: {len(headers)} → {headers}")

    # 한글 헤더 → DB 컬럼명 매핑
    # (화면 헤더가 COLUMNS_ORDERED와 다를 수 있으므로 명시적 매핑)
    HEADER_TO_COL = {
        '부서':         '부서',
        '담당자':       '담당자',
        '거래처':       '거래처',
        '거래처명':     '거래처명',
        '계약번호':     '계약번호',
        '계약구분':     '계약구분',
        '진행상태':     '진행상태_코드',  # 첫 번째 진행상태 = 코드 (1/X/Y)
        '약정/할인':    '약정할인',
        '총계약금액':   '총계약금액',
        '총이용금액':   '총이용금액',
        '총계약잔액':   '총계약잔액',
        '제품계약금액': '제품계약금액',
        '제품이용금액': '제품이용금액',
        '제품잔액':     '제품잔액',
        '장비계약금액': '장비계약금액',
        '장비이용금액': '장비이용금액',
        '남은장비잔액': '남은장비잔액',
        '재료계약금액': '재료계약금액',
        '재료이용금액': '재료이용금액',
        '남은재료잔액': '남은재료잔액',
    }
    # 진행상태가 2번 나올 경우 마지막은 텍스트(진행/완료/취소)
    # → 순서 기반으로 처리하기 위해 인덱스 추적

    # 데이터 줄 파싱
    rows = []
    진행상태_indices = [i for i, h in enumerate(headers) if h == '진행상태']

    for line in lines[header_idx + 1:]:
        stripped = line.strip()
        # 구분선 또는 빈 줄 스킵
        if not stripped:
            continue
        if all(c in '-|+ ' for c in stripped):
            continue
        if '|' not in line:
            continue

        # split('|') → 앞뒤 빈 요소 제거
        parts = line.split('|')
        parts = parts[1:-1]  # 줄 양끝 | 때문에 생기는 빈 요소 제거

        # 컬럼 수 맞추기
        while len(parts) < len(headers):
            parts.append('')

        # trim 후 빈 값은 None
        vals = [v.strip() if v.strip() else None for v in parts]
        raw  = dict(zip(headers, vals[:len(headers)]))

        # 계약번호 없으면 스킵
        if not raw.get('계약번호'):
            continue

        # 한글 헤더 → DB 컬럼명 변환 + 중복 '진행상태' 처리
        record = {}
        seen_진행상태 = 0
        for idx_h, h in enumerate(headers):
            val = vals[idx_h] if idx_h < len(vals) else None
            if h == '진행상태':
                if seen_진행상태 == 0:
                    record['진행상태_코드'] = val   # 첫 번째: 코드 (1/X/Y)
                else:
                    record['진행상태'] = val         # 두 번째: 텍스트 (진행/완료/취소)
                seen_진행상태 += 1
            else:
                col = HEADER_TO_COL.get(h, h)
                if col in NUMERIC_COLS:
                    record[col] = _clean_number(val) if val else 0.0
                else:
                    record[col] = val

        # 누락된 숫자 컬럼 기본값 처리
        for col in NUMERIC_COLS:
            if col not in record:
                record[col] = 0.0

        record['기준일자']     = BASE_DATE
        record['사업장']       = plant_code
        record['배치실행일시'] = run_dt
        rows.append(record)

    print(f"    ↳ 파싱 완료: {len(rows):,}건")
    return rows


# ─────────────────────────────────────────────
# DB 저장
# ─────────────────────────────────────────────
def get_db_conn():
    return pymysql.connect(
        host=DB_HOST, port=DB_PORT,
        user=DB_USER, password=DB_PASS,
        database=DB_NAME,
        charset='utf8mb4',
        autocommit=False,
    )


def save_to_db(rows, conn):
    if not rows:
        return 0

    cols = [
        '사업장', '부서', '담당자', '거래처', '거래처명', '계약번호', '계약구분',
        '진행상태_코드', '약정할인',
        '총계약금액', '총이용금액', '총계약잔액',
        '제품계약금액', '제품이용금액', '제품잔액',
        '장비계약금액', '장비이용금액', '남은장비잔액',
        '재료계약금액', '재료이용금액', '남은재료잔액',
        '진행상태', '기준일자', '배치실행일시',
    ]

    placeholders = ', '.join(['%s'] * len(cols))
    col_list     = ', '.join([f'`{c}`' for c in cols])
    # UPSERT: 기준일자+계약번호 중복이면 전체 업데이트
    update_clause = ', '.join([f'`{c}`=VALUES(`{c}`)' for c in cols if c not in ('기준일자', '계약번호')])

    sql = (
        f"INSERT INTO `{DB_TABLE}` ({col_list}) "
        f"VALUES ({placeholders}) "
        f"ON DUPLICATE KEY UPDATE {update_clause}"
    )

    total = 0
    with conn.cursor() as cur:
        batch = []
        for row in rows:
            batch.append(tuple(row.get(c) for c in cols))
            if len(batch) >= BATCH_SIZE:
                cur.executemany(sql, batch)
                total += len(batch)
                batch = []
        if batch:
            cur.executemany(sql, batch)
            total += len(batch)
    conn.commit()
    return total


# ─────────────────────────────────────────────
# 결과 없음 확인 (팝업 또는 상태바)
# ─────────────────────────────────────────────
def has_no_data(session):
    """데이터 없음 상태 감지"""
    try:
        status = session.findById("wnd[0]/sbar").text
        keywords = ['데이터 없음', 'No data', '선택된 데이터 없음', '조회 결과가 없습니다']
        if any(k in status for k in keywords):
            return True
    except Exception:
        pass

    # 팝업창 확인
    try:
        popup = session.findById("wnd[1]")
        if popup:
            try:
                session.findById("wnd[1]/tbar[0]/btn[0]").press()
            except Exception:
                pass
            return True
    except Exception:
        pass

    return False


# ─────────────────────────────────────────────
# 메인
# ─────────────────────────────────────────────
def main():
    print("=" * 60)
    print(f" SAP {SAP_TCODE} 자동화 시작")
    print(f" 기준일자: {BASE_DATE}")
    print(f" 사업장 수: {len(PLANT_CODES)}개")
    print("=" * 60)

    session = connect_sap()
    conn    = get_db_conn()
    run_dt  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    total_saved  = 0
    total_skipped = 0
    failed_plants = []

    for idx, plant in enumerate(PLANT_CODES, 1):
        print(f"\n[{idx:3d}/{len(PLANT_CODES)}] 사업장: {plant}")
        try:
            navigate_to_tcode(session, SAP_TCODE)
            input_conditions(session, plant)
            execute_report(session)

            if has_no_data(session):
                print(f"    ↳ 데이터 없음 → 스킵")
                total_skipped += 1
                continue

            text = extract_alv_to_clipboard(session)
            if not text:
                total_skipped += 1
                continue

            rows = parse_clipboard_text(text, plant, run_dt)
            if not rows:
                print(f"    ↳ 파싱 결과 0건 → 스킵")
                total_skipped += 1
                continue

            saved = save_to_db(rows, conn)
            total_saved += saved
            print(f"    ↳ 저장: {saved:,}건  (누계: {total_saved:,}건)")

        except Exception as e:
            print(f"    ↳ ❌ 오류: {e}")
            failed_plants.append((plant, str(e)))
            # 오류 시 T-Code 재이동 시도
            try:
                navigate_to_tcode(session, SAP_TCODE)
            except Exception:
                pass
            continue

    conn.close()

    print("\n" + "=" * 60)
    print(f" 완료!")
    print(f" 총 저장: {total_saved:,}건")
    print(f" 스킵:    {total_skipped}개 사업장")
    if failed_plants:
        print(f" 실패:    {len(failed_plants)}개 사업장")
        for p, err in failed_plants:
            print(f"   - {p}: {err}")
    print("=" * 60)


if __name__ == "__main__":
    main()


 SAP ZSDR0026 자동화 시작
 기준일자: 20260315
 사업장 수: 164개
[SAP] 연결 성공: OKP

[  1/164] 사업장: 1010
    ↳ ALV RowCount=3245
    ↳ 헤더 컬럼 수: 21 → ['부서', '담당자', '거래처', '거래처명', '계약번호', '계약구분', '진행상태', '약정/할인', '총계약금액', '총이용금액', '총계약잔액', '제품계약금액', '제품이용금액', '제품잔액', '장비계약금액', '장비이용금액', '남은장비잔액', '재료계약금액', '재료이용금액', '남은재료잔액', '진행상태']
    ↳ 파싱 완료: 3,245건
    ↳ 저장: 3,245건  (누계: 3,245건)

[  2/164] 사업장: 1020
    ↳ ALV RowCount=36
    ↳ 헤더 컬럼 수: 21 → ['부서', '담당자', '거래처', '거래처명', '계약번호', '계약구분', '진행상태', '약정/할인', '총계약금액', '총이용금액', '총계약잔액', '제품계약금액', '제품이용금액', '제품잔액', '장비계약금액', '장비이용금액', '남은장비잔액', '재료계약금액', '재료이용금액', '남은재료잔액', '진행상태']
    ↳ 파싱 완료: 36건
    ↳ 저장: 36건  (누계: 3,281건)

[  3/164] 사업장: 1030
    ↳ ALV RowCount=4047
    ↳ 헤더 컬럼 수: 21 → ['부서', '담당자', '거래처', '거래처명', '계약번호', '계약구분', '진행상태', '약정/할인', '총계약금액', '총이용금액', '총계약잔액', '제품계약금액', '제품이용금액', '제품잔액', '장비계약금액', '장비이용금액', '남은장비잔액', '재료계약금액', '재료이용금액', '남은재료잔액', '진행상태']
    ↳ 파싱 완료: 4,047건
    ↳ 저장: 4,047건  (누계: 7,328건)

[  4/164] 사업장: 1040
    ↳ ALV RowCoun